In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# === LOAD DATASET ===
df = pd.read_csv('../datasets/insurance.csv')

print("Dataset 4 shape:", df.shape)

# ---- CREATE CLASSIFICATION TARGET ----
# charges is continuous → convert to binary
threshold = df['charges'].median()
df['high_cost'] = (df['charges'] > threshold).astype(int)

y = df['high_cost']
X = df.drop(columns=['charges', 'high_cost'])

# Handle missing
for col in X.select_dtypes(include=[np.number]).columns:
    X[col].fillna(X[col].median(), inplace=True)

# Encode categorical
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col])

# Normalize
X_scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)

print("Cleaned:", X_scaled.shape)

# === TRAIN MODELS ===
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]

    results[name] = {
        'Accuracy': round(accuracy_score(y_test, pred), 4),
        'F1 Score': round(f1_score(y_test, pred), 4),
        'ROC-AUC': round(roc_auc_score(y_test, proba), 4)
    }

results_df = pd.DataFrame(results).T

print("\nDataset 4 Results:")
print(results_df)

results_df.to_csv('../results/insurance_results.csv')
print("Dataset 4 done!")


Dataset 4 shape: (1338, 7)
Cleaned: (1338, 6)

Dataset 4 Results:
                     Accuracy  F1 Score  ROC-AUC
Logistic Regression    0.9104    0.9111   0.9433
Random Forest          0.9440    0.9421   0.9481
XGBoost                0.9328    0.9313   0.9522
Dataset 4 done!
